# Basic Pitch Note-Capture Experiments V3 — Two-Pass + Cluster Pruning + Timing Diagnostics

This version is built from the working fixed-path V2 notebook, keeping the same Drive/GuitarSet path and pairing logic.

V3 adds experiments that target the real V2 bottleneck:

- **two-pass detection**: high-precision pass + recall pass, then merge carefully
- **onset-cluster pruning**: keep physically plausible near-simultaneous note clusters
- **recording-level adaptive thresholds**: solo/comp or noisy/sparse behavior
- **onset tolerance diagnostics**: test whether misses are timing errors vs true detection misses
- **val/test discipline helpers**: use validation for choosing settings and test for final reporting


In [1]:
from pathlib import Path
from collections import defaultdict
import hashlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)



In [2]:
# Mount Google Drive when running in Colab.
# This must run before any /content/drive/MyDrive/... paths are used.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
except ModuleNotFoundError:
    print('Not running in Colab. Skipping Google Drive mount.')



Mounted at /content/drive
Google Drive mounted successfully.


## 1. Configuration

This uses the same path assumptions as the working notebook:

```text
/content/drive/MyDrive/Capstone/FullGuitarSetData
/content/drive/MyDrive/FullGuitarSetData
```

Expected data structure:

```text
FullGuitarSetData/
├── JamsFiles/
└── AudioFiles/
```

Experiment outputs are saved to:

```text
/content/drive/MyDrive/Capstone/outputs/basic_pitch_note_capture_experiments/
```


In [3]:
# -------------------------
# USER CONFIG - fixed-path V3
# -------------------------

CAPSTONE_ROOT = Path('/content/drive/MyDrive/Capstone')
OUTPUT_DIR = CAPSTONE_ROOT / 'outputs' / 'basic_pitch_note_capture_experiments_v3'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Capstone/FullGuitarSetData'),
    Path('/content/drive/MyDrive/FullGuitarSetData'),
    Path('/content/drive/MyDrive/Capstone/GuitarSet'),
    Path('/content/drive/MyDrive/GuitarSet'),
    Path('/content/drive/MyDrive/Capstone'),
    Path('/content/drive/MyDrive'),
    Path('/mnt/data/fretwork_repo/GuitarSet'),
    Path('/mnt/data/fretwork_repo'),
]

AUDIO_ROOT_CANDIDATES = [
    CAPSTONE_ROOT / 'GuitarSet' / 'Audio',
    CAPSTONE_ROOT / 'GuitarSet' / 'AudioFiles',
    CAPSTONE_ROOT / 'FullGuitarSetData' / 'AudioFiles',
    CAPSTONE_ROOT / 'FullGuitarSetData' / 'Audio',
    CAPSTONE_ROOT / 'Audio',
    CAPSTONE_ROOT / 'GuitarSet',
    CAPSTONE_ROOT,
    Path('/content/drive/MyDrive/GuitarSet/Audio'),
    Path('/content/drive/MyDrive/GuitarSet'),
]

AUDIO_EXTENSIONS = ['.wav', '.mp3', '.m4a', '.flac', '.ogg']

# Guitar standard tuning range. Keep this wide enough for bends/high frets.
DEFAULT_MIN_MIDI = 40   # low E2
DEFAULT_MAX_MIDI = 88   # high guitar-ish range
OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64]  # E2, A2, D3, G3, B3, E4

# Matching predicted Basic Pitch notes to GuitarSet GT notes.
AUDIO_MATCH_ONSET_TOLERANCE_SECONDS = 0.05

# Use held-out test split like the original notebook.
USE_HELDOUT_SPLIT = True
SPLIT_SEED = 42
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

# Debug small, then set to None for full test set.
MAX_AUDIO_RECORDINGS = None

# Variant set controls. 'core' includes V2 + V3 core variants. 'full' adds heavier preprocessing variants.
VARIANT_SET = 'core'  # options: 'core', 'full', 'threshold_sweep'

# Basic Pitch preprocessing cache.
CACHE_DIR = OUTPUT_DIR / 'cache'
PREPROCESSED_AUDIO_DIR = CACHE_DIR / 'preprocessed_audio'
BASIC_PITCH_NOTE_CACHE_DIR = CACHE_DIR / 'basic_pitch_notes'
for d in [CACHE_DIR, PREPROCESSED_AUDIO_DIR, BASIC_PITCH_NOTE_CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('OUTPUT_DIR:', OUTPUT_DIR.resolve())
print('VARIANT_SET:', VARIANT_SET)
print('\nData root candidates:')
for p in DATA_ROOT_CANDIDATES:
    print(f' - {p} | exists: {p.exists()}')
print('\nAudio root candidates:')
for p in AUDIO_ROOT_CANDIDATES:
    print(f' - {p} | exists: {p.exists()}')




OUTPUT_DIR: /content/drive/.shortcut-targets-by-id/1JNqe8bukG93wCWVxbk7SKlNvVyIZHyTC/Capstone/outputs/basic_pitch_note_capture_experiments_v3
VARIANT_SET: core

Data root candidates:
 - /content/drive/MyDrive/Capstone/FullGuitarSetData | exists: True
 - /content/drive/MyDrive/FullGuitarSetData | exists: False
 - /content/drive/MyDrive/Capstone/GuitarSet | exists: True
 - /content/drive/MyDrive/GuitarSet | exists: False
 - /content/drive/MyDrive/Capstone | exists: True
 - /content/drive/MyDrive | exists: True
 - /mnt/data/fretwork_repo/GuitarSet | exists: False
 - /mnt/data/fretwork_repo | exists: False

Audio root candidates:
 - /content/drive/MyDrive/Capstone/GuitarSet/Audio | exists: True
 - /content/drive/MyDrive/Capstone/GuitarSet/AudioFiles | exists: False
 - /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles | exists: True
 - /content/drive/MyDrive/Capstone/FullGuitarSetData/Audio | exists: False
 - /content/drive/MyDrive/Capstone/Audio | exists: True
 - /content/drive/

## 2. Install / import Basic Pitch environment

This is copied from the working notebook style and includes the Python 3.12 `pkgutil.ImpImporter` patch used for Colab.

Run this once after restarting the runtime.


In [4]:
# ============================================================
# FIX-ALL Basic Pitch install/import cell for Colab Python 3.12
# Run this ONCE after Runtime -> Restart runtime
# ============================================================

import importlib
import pkgutil
import zipimport

print('Python:', sys.version)

def run(cmd):
    print('\n$', ' '.join(cmd))
    subprocess.check_call(cmd)

# Some Colab/system pkg_resources versions expect pkgutil.ImpImporter,
# which was removed in Python 3.12.
if not hasattr(pkgutil, 'ImpImporter'):
    pkgutil.ImpImporter = zipimport.zipimporter

run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'wheel', 'setuptools==80.9.0'])

run([sys.executable, '-m', 'pip', 'install', '-q',
     'librosa>=0.10',
     'soundfile',
     'mir-eval',
     'pretty_midi',
     'resampy==0.4.2',
     'onnxruntime',
     'scipy'])

run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'basic-pitch==0.4.0'])

if not hasattr(pkgutil, 'ImpImporter'):
    pkgutil.ImpImporter = zipimport.zipimporter

import librosa
import soundfile as sf
from scipy import signal
from basic_pitch.inference import predict as basic_pitch_predict

print('\n✅ Basic Pitch import successful.')
print('✅ librosa:', librosa.__version__)
print('✅ soundfile import successful.')
print('✅ scipy signal import successful.')
print('✅ onnxruntime installed.')



Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

$ /usr/bin/python3 -m pip install -q --upgrade pip wheel setuptools==80.9.0

$ /usr/bin/python3 -m pip install -q librosa>=0.10 soundfile mir-eval pretty_midi resampy==0.4.2 onnxruntime scipy

$ /usr/bin/python3 -m pip install -q --no-deps basic-pitch==0.4.0



✅ Basic Pitch import successful.
✅ librosa: 0.11.0
✅ soundfile import successful.
✅ scipy signal import successful.
✅ onnxruntime installed.


## 3. GuitarSet JAMS parsing

This section reads GuitarSet `.jams` files directly without requiring the external `jams` package. Ground truth is used only for scoring Basic Pitch note capture.


In [5]:
def find_jams_dir(data_root):
    """Return a directory containing .jams files under data_root, or None if not found."""
    candidates = [
        data_root / 'JamsFiles',
        data_root / 'Annotations',
        data_root / 'GuitarSet' / 'Annotations',
        data_root / 'FullGuitarSetData' / 'JamsFiles',
        data_root / 'FullGuitarSetData' / 'Annotations',
        data_root,
    ]
    for c in candidates:
        if c.exists() and list(c.glob('*.jams')):
            return c

    if data_root.exists():
        try:
            for match in data_root.rglob('*.jams'):
                return match.parent
        except Exception as e:
            print(f'Could not recursively search {data_root}: {e}')
    return None


def choose_data_root_and_jams_dir(candidates):
    checked = []
    for root in candidates:
        checked.append((root, root.exists()))
        if not root.exists():
            continue
        jams_dir = find_jams_dir(root)
        if jams_dir is not None:
            return root, jams_dir

    print('Could not find .jams files automatically.')
    print('Checked these DATA_ROOT_CANDIDATES:')
    for root, exists in checked:
        print(f' - {root} | exists: {exists}')
    raise FileNotFoundError(
        'Could not find any .jams files. Add the correct GuitarSet/FullGuitarSetData path to DATA_ROOT_CANDIDATES.'
    )


def get_annotation_data(annotation):
    """Handle both list-form and column-dict-form JAMS annotation data."""
    data = annotation.get('data', [])
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        keys = ['time', 'duration', 'value', 'confidence']
        n = len(data.get('time', []))
        return [{k: data.get(k, [None] * n)[i] for k in keys} for i in range(n)]
    return []


def parse_string_from_data_source(data_source):
    """GuitarSet stores each string as a separate note_midi annotation with data_source 0-5."""
    try:
        s = int(data_source)
        return s if 0 <= s <= 5 else None
    except Exception:
        return None


def parse_jams_file(path):
    """Parse GuitarSet note_midi annotations for GT pitch/onset scoring."""
    with open(path, 'r') as f:
        jam = json.load(f)

    notes = []
    for ann in jam.get('annotations', []):
        ns = ann.get('namespace')
        if ns != 'note_midi':
            continue
        rows = get_annotation_data(ann)
        data_source = ann.get('annotation_metadata', {}).get('data_source', '')
        inferred_string = parse_string_from_data_source(data_source)

        for r in rows:
            v = r.get('value')
            if isinstance(v, dict):
                midi = v.get('midi_note') or v.get('note') or v.get('pitch')
                string = v.get('string', inferred_string)
                fret = v.get('fret')
            else:
                midi = v
                string = inferred_string
                fret = None

            if midi is None:
                continue

            midi_int = int(round(float(midi)))
            if string is not None and fret is None:
                fret = midi_int - OPEN_STRING_MIDI[int(string)]

            start = float(r.get('time', 0.0))
            duration = float(r.get('duration', 0.0) or 0.0)
            notes.append({
                'recording': Path(path).stem,
                'start': start,
                'duration': duration,
                'end': start + duration,
                'midi': midi_int,
                'pitch_class': midi_int % 12,
                'true_string': None if string is None else int(string),
                'true_fret': None if fret is None else int(round(float(fret))),
                'source': data_source,
            })

    notes = sorted(notes, key=lambda x: (x['start'], x['midi']))
    return {'recording': path.stem, 'path': str(path), 'notes': notes}


DATA_ROOT, JAMS_DIR = choose_data_root_and_jams_dir(DATA_ROOT_CANDIDATES)
JAMS_FILES = sorted(JAMS_DIR.glob('*.jams'))
print(f'DATA_ROOT selected: {DATA_ROOT}')
print(f'Found {len(JAMS_FILES)} JAMS files in {JAMS_DIR}')
print('\n'.join(p.name for p in JAMS_FILES[:10]))

records = [parse_jams_file(p) for p in JAMS_FILES]
records = [r for r in records if len(r['notes']) > 0]
print('Parsed records with notes:', len(records))
if records:
    print('Example record:', records[0]['recording'])
    print('Notes:', len(records[0]['notes']))
    display(pd.DataFrame(records[0]['notes']).head())
else:
    raise ValueError('No records parsed. Check JAMS_FILES and DATA_ROOT_CANDIDATES.')


DATA_ROOT selected: /content/drive/MyDrive/Capstone/FullGuitarSetData
Found 360 JAMS files in /content/drive/MyDrive/Capstone/FullGuitarSetData/JamsFiles
00_BN1-129-Eb_comp.jams
00_BN1-129-Eb_solo.jams
00_BN1-147-Gb_comp.jams
00_BN1-147-Gb_solo.jams
00_BN2-131-B_comp.jams
00_BN2-131-B_solo.jams
00_BN2-166-Ab_comp.jams
00_BN2-166-Ab_solo.jams
00_BN3-119-G_comp.jams
00_BN3-119-G_solo.jams
Parsed records with notes: 360
Example record: 00_BN1-129-Eb_comp
Notes: 133


,recording,start,duration,end,midi,pitch_class,true_string,true_fret,source
0,00_BN1-129-Eb_comp,0.048816,0.423764,0.472580,51,3,1,6,1
1,00_BN1-129-Eb_comp,0.049791,0.452789,0.502580,65,5,4,6,4
2,00_BN1-129-Eb_comp,0.052717,0.458594,0.511311,62,2,3,7,3
3,00_BN1-129-Eb_comp,0.519995,0.417959,0.937955,51,3,1,6,1
4,00_BN1-129-Eb_comp,0.722036,0.859138,1.581175,58,10,2,8,2


In [6]:
def split_records_by_recording(records, train_frac=0.70, val_frac=0.15, test_frac=0.15, seed=42):
    """Same split style as original notebook: split by recording, not individual note."""
    if not np.isclose(train_frac + val_frac + test_frac, 1.0):
        raise ValueError('train_frac + val_frac + test_frac must sum to 1.0')

    rng = random.Random(seed)
    groups = {
        'solo': [r for r in records if r['recording'].endswith('_solo')],
        'comp': [r for r in records if r['recording'].endswith('_comp')],
        'other': [r for r in records if not (r['recording'].endswith('_solo') or r['recording'].endswith('_comp'))],
    }

    train, val, test = [], [], []
    for label, group in groups.items():
        group = list(group)
        rng.shuffle(group)
        n = len(group)
        n_train = int(round(n * train_frac))
        n_val = int(round(n * val_frac))
        train.extend(group[:n_train])
        val.extend(group[n_train:n_train + n_val])
        test.extend(group[n_train + n_val:])

    return train, val, test

if USE_HELDOUT_SPLIT:
    TRAIN_RECORDS, VAL_RECORDS, TEST_RECORDS = split_records_by_recording(
        records, TRAIN_FRAC, VAL_FRAC, TEST_FRAC, seed=SPLIT_SEED
    )
else:
    TRAIN_RECORDS, VAL_RECORDS, TEST_RECORDS = records, [], records

print('Split sizes:')
print('TRAIN_RECORDS:', len(TRAIN_RECORDS))
print('VAL_RECORDS:', len(VAL_RECORDS))
print('TEST_RECORDS:', len(TEST_RECORDS))
print('Total:', len(TRAIN_RECORDS) + len(VAL_RECORDS) + len(TEST_RECORDS))



Split sizes:
TRAIN_RECORDS: 252
VAL_RECORDS: 54
TEST_RECORDS: 54
Total: 360


## 4. Pair held-out records with audio files

This uses the same stem matching logic as the working notebook:

```text
00_Jazz3-150-D_comp.jams      ↔ 00_Jazz3-150-D_comp_mic.wav
00_Jazz3-150-D_comp.jams      ↔ 00_Jazz3-150-D_comp_hex.wav
00_Jazz3-150-D_comp.jams      ↔ 00_Jazz3-150-D_comp.wav
```


In [ ]:
def find_audio_files(audio_roots, exts=AUDIO_EXTENSIONS):
    """Return dict stem -> path for audio files under candidate roots."""
    audio_by_stem = {}
    for root in audio_roots:
        root = Path(root)
        if not root.exists():
            continue
        for ext in exts:
            try:
                for p in root.rglob(f'*{ext}'):
                    audio_by_stem.setdefault(p.stem, p)
            except Exception as e:
                print(f'Could not search {root}: {e}')
    return audio_by_stem


def audio_candidates_for_recording(recording):
    recording = str(recording)
    cands = [
        recording,
        f'{recording}_mic',
        f'{recording}_hex',
        recording.replace('_solo', '_solo_mic'),
        recording.replace('_comp', '_comp_mic'),
        recording.replace('_solo', '_solo_hex'),
        recording.replace('_comp', '_comp_hex'),
    ]
    seen, out = set(), []
    for c in cands:
        if c not in seen:
            out.append(c)
            seen.add(c)
    return out


def find_audio_for_record(record):
    rec = record['recording']
    for stem in audio_candidates_for_recording(rec):
        if stem in AUDIO_BY_STEM:
            return AUDIO_BY_STEM[stem]
    return None

AUDIO_BY_STEM = find_audio_files(AUDIO_ROOT_CANDIDATES)
print(f'Found {len(AUDIO_BY_STEM)} audio files.')
for k, v in list(AUDIO_BY_STEM.items())[:15]:
    print(' -', k, '->', v)

AUDIO_EVAL_RECORDS = TEST_RECORDS if USE_HELDOUT_SPLIT else records
AUDIO_EVAL_LABEL = 'heldout_test_audio' if USE_HELDOUT_SPLIT else 'all_records_audio_exploratory'

paired_records = []
missing_audio = []
for record in AUDIO_EVAL_RECORDS:
    audio_path = find_audio_for_record(record)
    if audio_path is None:
        missing_audio.append(record['recording'])
    else:
        paired_records.append((record, audio_path))

print(f'Audio eval set: {AUDIO_EVAL_LABEL}')
print(f'Paired {len(paired_records)} / {len(AUDIO_EVAL_RECORDS)} eval records with audio files.')
for rec, ap in paired_records[:15]:
    print(' -', rec['recording'], '->', ap.name)

if missing_audio:
    print(f'\nMissing audio for {len(missing_audio)} eval records. First few:')
    print(missing_audio[:25])

if not paired_records:
    print('\nNo pairs found. Check AUDIO_ROOT_CANDIDATES and whether audio stems use _mic/_hex suffixes.')



Found 636 audio files.
 - 00_BN1-129-Eb_comp_mic -> /content/drive/MyDrive/Capstone/GuitarSet/Audio/00_BN1-129-Eb_comp_mic.wav
 - 00_Jazz1-130-D_solo_mic -> /content/drive/MyDrive/Capstone/GuitarSet/Audio/00_Jazz1-130-D_solo_mic.wav
 - 00_Funk1-97-C_solo_mic -> /content/drive/MyDrive/Capstone/GuitarSet/Audio/00_Funk1-97-C_solo_mic.wav
 - 00_Funk1-97-C_comp_mic -> /content/drive/MyDrive/Capstone/GuitarSet/Audio/00_Funk1-97-C_comp_mic.wav
 - 00_Rock1-90-C#_comp_mic -> /content/drive/MyDrive/Capstone/GuitarSet/Audio/00_Rock1-90-C#_comp_mic.wav
 - 00_Rock1-90-C#_solo_mic -> /content/drive/MyDrive/Capstone/GuitarSet/Audio/00_Rock1-90-C#_solo_mic.wav
 - 00_Jazz1-130-D_comp_mic -> /content/drive/MyDrive/Capstone/GuitarSet/Audio/00_Jazz1-130-D_comp_mic.wav
 - 00_SS1-68-E_solo_mic -> /content/drive/MyDrive/Capstone/GuitarSet/Audio/00_SS1-68-E_solo_mic.wav
 - 00_SS1-68-E_comp_mic -> /content/drive/MyDrive/Capstone/GuitarSet/Audio/00_SS1-68-E_comp_mic.wav
 - 00_BN1-129-Eb_solo_mic -> /content/dri

## 5. Basic Pitch preprocessing and postprocessing variants

Each experiment variant has two stages:

1. Optional **audio preprocessing** saved as a temporary WAV, such as bandpass filtering or harmonic-only separation.
2. Optional **note postprocessing**, such as amplitude thresholding, minimum duration filtering, same-pitch merge, and MIDI range filtering.

The original audio file is never modified.


In [ ]:

def midi_to_note_name_simple(midi):
    names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    midi = int(round(midi))
    return f"{names[midi % 12]}{midi // 12 - 1}"


def _variant_hash(variant):
    s = json.dumps(variant, sort_keys=True, default=str)
    return hashlib.md5(s.encode('utf-8')).hexdigest()[:10]


def _safe_audio_stem(audio_path):
    return Path(audio_path).stem.replace('/', '_').replace(' ', '_')


def normalize_peak(y, target_peak=0.98):
    peak = float(np.max(np.abs(y))) if len(y) else 0.0
    if peak <= 1e-9:
        return y
    return y / peak * target_peak


def apply_band_filter(y, sr, low_hz=None, high_hz=None, order=4):
    """Apply highpass, lowpass, or bandpass filter using SOS for stability."""
    nyq = sr / 2.0
    if low_hz is not None and high_hz is not None:
        low = max(float(low_hz) / nyq, 1e-5)
        high = min(float(high_hz) / nyq, 0.999)
        if low >= high:
            return y
        sos = signal.butter(order, [low, high], btype='bandpass', output='sos')
    elif low_hz is not None:
        low = max(float(low_hz) / nyq, 1e-5)
        sos = signal.butter(order, low, btype='highpass', output='sos')
    elif high_hz is not None:
        high = min(float(high_hz) / nyq, 0.999)
        sos = signal.butter(order, high, btype='lowpass', output='sos')
    else:
        return y
    return signal.sosfiltfilt(sos, y).astype(np.float32)


def preprocess_audio_for_variant(audio_path, variant, use_cache=True):
    """Return path to audio file for Basic Pitch: original or cached preprocessed WAV."""
    preprocess = variant.get('preprocess', {}) or {}
    if not preprocess:
        return Path(audio_path)

    audio_path = Path(audio_path)
    out_name = f"{_safe_audio_stem(audio_path)}__{variant['name']}__{_variant_hash(variant)}.wav"
    out_path = PREPROCESSED_AUDIO_DIR / out_name
    if use_cache and out_path.exists():
        return out_path

    sr = int(preprocess.get('sr', 22050))
    y, sr = librosa.load(str(audio_path), sr=sr, mono=True)

    if preprocess.get('trim_silence', False):
        top_db = float(preprocess.get('trim_top_db', 35))
        y, _ = librosa.effects.trim(y, top_db=top_db)

    if preprocess.get('harmonic_only', False):
        y_harm, _ = librosa.effects.hpss(y)
        y = y_harm

    if preprocess.get('hpss_mix', False):
        # Less aggressive than harmonic_only: keep some attack/transient information.
        y_harm, y_perc = librosa.effects.hpss(y)
        h_w = float(preprocess.get('harmonic_weight', 1.0))
        p_w = float(preprocess.get('percussive_weight', 0.25))
        y = h_w * y_harm + p_w * y_perc

    low_hz = preprocess.get('low_hz')
    high_hz = preprocess.get('high_hz')
    if low_hz is not None or high_hz is not None:
        y = apply_band_filter(y, sr, low_hz=low_hz, high_hz=high_hz, order=int(preprocess.get('filter_order', 4)))

    if preprocess.get('preemphasis', False):
        coef = float(preprocess.get('preemphasis_coef', 0.97))
        y = librosa.effects.preemphasis(y, coef=coef)

    if preprocess.get('normalize_rms', False):
        target_rms = float(preprocess.get('target_rms', 0.07))
        rms = float(np.sqrt(np.mean(y ** 2))) if len(y) else 0.0
        if rms > 1e-9:
            y = y * (target_rms / rms)

    if preprocess.get('normalize_peak', True):
        y = normalize_peak(y, target_peak=float(preprocess.get('target_peak', 0.98)))

    sf.write(str(out_path), y.astype(np.float32), sr)
    return out_path


def raw_basic_pitch_events(audio_for_bp, variant=None, use_cache=True, cache_key=None):
    """Run Basic Pitch and cache the raw note events before threshold/range filtering.

    V2 supports Basic Pitch's own inference knobs through variant['bp'], for example:
    {'bp': {'onset_threshold': 0.4, 'frame_threshold': 0.3, 'minimum_note_length': 50}}
    """
    variant = variant or {}
    audio_for_bp = Path(audio_for_bp)
    bp_kwargs = dict(variant.get('bp', {}) or {})
    key = cache_key or (_safe_audio_stem(audio_for_bp) + '__' + _variant_hash({'bp': bp_kwargs}))
    cache_path = BASIC_PITCH_NOTE_CACHE_DIR / f'{key}_raw_events.csv'

    if use_cache and cache_path.exists():
        return pd.read_csv(cache_path).to_dict('records')

    print(f'Running Basic Pitch on: {audio_for_bp.name} | bp_kwargs={bp_kwargs}')
    try:
        _, _, note_events = basic_pitch_predict(str(audio_for_bp), **bp_kwargs)
    except TypeError as e:
        print('Basic Pitch did not accept one of these kwargs:', bp_kwargs)
        print('Falling back to default predict(). Error was:', repr(e))
        _, _, note_events = basic_pitch_predict(str(audio_for_bp))

    rows = []
    for event in note_events:
        start, end, pitch_midi, amplitude = event[0], event[1], event[2], event[3]
        midi = int(round(float(pitch_midi)))
        rows.append({
            'start': float(start),
            'end': float(end),
            'duration': float(end - start),
            'midi': midi,
            'pitch_class': midi % 12,
            'note_name': midi_to_note_name_simple(midi),
            'amplitude': float(amplitude),
            'source': 'basic_pitch_raw',
        })

    df = pd.DataFrame(rows).sort_values(['start', 'midi']) if rows else pd.DataFrame(rows)
    df.to_csv(cache_path, index=False)
    return rows


def resolve_amplitude_threshold(raw_events, variant):
    """Fixed threshold by default; optional per-record adaptive threshold for noisy/quiet recordings."""
    fixed = variant.get('amplitude_threshold', 0.30)
    if not variant.get('adaptive_amplitude', False):
        return float(fixed)

    amps = np.array([float(e.get('amplitude', 0.0)) for e in raw_events], dtype=float)
    amps = amps[np.isfinite(amps)]
    if len(amps) == 0:
        return float(fixed)

    q = float(variant.get('adaptive_quantile', 0.25))
    floor = float(variant.get('adaptive_floor', 0.20))
    ceiling = float(variant.get('adaptive_ceiling', 0.50))
    threshold = float(np.quantile(amps, q))
    threshold = max(floor, min(ceiling, threshold))
    return threshold


def merge_same_pitch_notes(notes, max_gap_seconds=0.08):
    """Merge adjacent same-MIDI notes when Basic Pitch splits one sustained note.

    This was too aggressive in v1, so v2 keeps it available but does not use it in the recommended core set.
    """
    if not notes:
        return []
    notes_sorted = sorted(notes, key=lambda n: (float(n['start']), int(n['midi'])))
    merged = []
    current = dict(notes_sorted[0])

    for n in notes_sorted[1:]:
        same_pitch = int(n['midi']) == int(current['midi'])
        gap = float(n['start']) - float(current.get('end', current['start'] + current.get('duration', 0.0)))
        if same_pitch and gap <= max_gap_seconds:
            current_end = max(float(current.get('end', current['start'] + current.get('duration', 0.0))), float(n.get('end', n['start'] + n.get('duration', 0.0))))
            current['end'] = current_end
            current['duration'] = current_end - float(current['start'])
            current['amplitude'] = max(float(current.get('amplitude', 0.0)), float(n.get('amplitude', 0.0)))
        else:
            merged.append(current)
            current = dict(n)
    merged.append(current)
    return merged


def drop_near_duplicate_same_pitch(notes, window_seconds=0.025):
    """Conservative de-duplication: if two same-pitch notes start almost together, keep the louder one.

    Unlike merging, this does not join notes across longer gaps, so it is safer for repeated guitar picking.
    """
    if not notes:
        return []
    by_pitch = defaultdict(list)
    for n in notes:
        by_pitch[int(n['midi'])].append(dict(n))

    kept = []
    for midi, group in by_pitch.items():
        group = sorted(group, key=lambda x: float(x['start']))
        cluster = [group[0]]
        for n in group[1:]:
            if float(n['start']) - float(cluster[-1]['start']) <= window_seconds:
                cluster.append(n)
            else:
                kept.append(max(cluster, key=lambda x: float(x.get('amplitude', 0.0))))
                cluster = [n]
        kept.append(max(cluster, key=lambda x: float(x.get('amplitude', 0.0))))
    return sorted(kept, key=lambda n: (float(n['start']), int(n['midi'])))


def cap_polyphony_by_onset(notes, window_seconds=0.04, max_notes=6):
    """Keep only the loudest N notes in near-simultaneous onset clusters.

    This is most relevant for comp recordings or noisy output clusters. Guitar cannot physically play more than 6 notes at once.
    """
    if not notes:
        return []
    notes_sorted = sorted([dict(n) for n in notes], key=lambda n: float(n['start']))
    capped = []
    cluster = [notes_sorted[0]]
    cluster_start = float(notes_sorted[0]['start'])
    for n in notes_sorted[1:]:
        if float(n['start']) - cluster_start <= window_seconds:
            cluster.append(n)
        else:
            capped.extend(sorted(cluster, key=lambda x: float(x.get('amplitude', 0.0)), reverse=True)[:max_notes])
            cluster = [n]
            cluster_start = float(n['start'])
    capped.extend(sorted(cluster, key=lambda x: float(x.get('amplitude', 0.0)), reverse=True)[:max_notes])
    return sorted(capped, key=lambda n: (float(n['start']), int(n['midi'])))



def drop_local_weak_ghost_notes(notes, onset_window_seconds=0.05, local_amp_ratio=0.35):
    """Remove likely ghost notes that are weak relative to nearby simultaneous notes.

    Idea: Basic Pitch may output faint neighboring/harmonic notes around a real note onset.
    Within each onset neighborhood, keep notes that are at least local_amp_ratio of the strongest note.

    Conservative use:
    - local_amp_ratio=0.30 to 0.45
    - onset_window_seconds=0.04 to 0.06
    """
    if not notes:
        return []
    notes_sorted = sorted([dict(n) for n in notes], key=lambda n: (float(n['start']), int(n['midi'])))
    keep = []
    for i, n in enumerate(notes_sorted):
        start = float(n['start'])
        nearby = [m for m in notes_sorted if abs(float(m['start']) - start) <= onset_window_seconds]
        if len(nearby) <= 1:
            keep.append(n)
            continue
        max_amp = max(float(m.get('amplitude', 0.0)) for m in nearby)
        if max_amp <= 1e-9 or float(n.get('amplitude', 0.0)) >= local_amp_ratio * max_amp:
            keep.append(n)
    return sorted(keep, key=lambda n: (float(n['start']), int(n['midi'])))


def remove_harmonic_shadow_notes(notes, onset_window_seconds=0.045, shadow_amp_ratio=0.45, intervals=(12, 19, 24)):
    """Remove weak harmonic/octave shadow notes near a stronger note onset.

    Common artifacts:
    - 12 semitones = octave
    - 19 semitones = octave + fifth
    - 24 semitones = two octaves

    If a note is close in onset, separated by one of these intervals, and much weaker
    than the other note, it is treated as a possible harmonic shadow.
    """
    if not notes:
        return []
    notes_sorted = sorted([dict(n) for n in notes], key=lambda n: (float(n['start']), int(n['midi'])))
    intervals = set(int(x) for x in intervals)
    drop_idx = set()

    for i, n in enumerate(notes_sorted):
        if i in drop_idx:
            continue
        n_start = float(n['start'])
        n_midi = int(n['midi'])
        n_amp = float(n.get('amplitude', 0.0))
        for j, other in enumerate(notes_sorted):
            if i == j:
                continue
            close_onset = abs(n_start - float(other['start'])) <= onset_window_seconds
            harmonic_interval = abs(n_midi - int(other['midi'])) in intervals
            much_weaker = n_amp < shadow_amp_ratio * float(other.get('amplitude', 0.0))
            if close_onset and harmonic_interval and much_weaker:
                drop_idx.add(i)
                break

    return [n for i, n in enumerate(notes_sorted) if i not in drop_idx]


def remove_isolated_low_confidence_notes(notes, context_window_seconds=0.25, isolated_amp_threshold=0.38):
    """Remove low-confidence notes that are isolated from nearby predicted notes.

    This targets small random blips. It is intentionally conservative: high-amplitude notes
    are kept even if isolated, because guitar solos can contain sparse notes.
    """
    if not notes:
        return []
    notes_sorted = sorted([dict(n) for n in notes], key=lambda n: (float(n['start']), int(n['midi'])))
    keep = []
    for i, n in enumerate(notes_sorted):
        amp = float(n.get('amplitude', 0.0))
        if amp >= isolated_amp_threshold:
            keep.append(n)
            continue
        start = float(n['start'])
        has_context = any(
            i != j and abs(float(m['start']) - start) <= context_window_seconds
            for j, m in enumerate(notes_sorted)
        )
        if has_context:
            keep.append(n)
    return sorted(keep, key=lambda n: (float(n['start']), int(n['midi'])))

def postprocess_basic_pitch_events(raw_events, variant):
    amplitude_threshold = resolve_amplitude_threshold(raw_events, variant)
    min_midi = int(variant.get('min_midi', DEFAULT_MIN_MIDI))
    max_midi = int(variant.get('max_midi', DEFAULT_MAX_MIDI))
    min_duration = float(variant.get('min_duration', 0.0) or 0.0)

    notes = []
    for e in raw_events:
        if float(e.get('amplitude', 0.0)) < amplitude_threshold:
            continue
        midi = int(round(float(e['midi'])))
        if midi < min_midi or midi > max_midi:
            continue
        duration = float(e.get('duration', 0.0) or 0.0)
        if duration < min_duration:
            continue
        notes.append({
            'start': float(e['start']),
            'end': float(e.get('end', float(e['start']) + duration)),
            'duration': duration,
            'midi': midi,
            'pitch_class': midi % 12,
            'note_name': midi_to_note_name_simple(midi),
            'amplitude': float(e.get('amplitude', 0.0)),
            'source': f"basic_pitch_{variant['name']}",
            'resolved_amplitude_threshold': amplitude_threshold,
        })

    if variant.get('dedupe_same_pitch_close', False):
        notes = drop_near_duplicate_same_pitch(notes, window_seconds=float(variant.get('dedupe_window_seconds', 0.025)))

    if variant.get('merge_same_pitch', False):
        notes = merge_same_pitch_notes(notes, max_gap_seconds=float(variant.get('merge_gap_seconds', 0.08)))

    if variant.get('ghost_local_prune', False):
        notes = drop_local_weak_ghost_notes(
            notes,
            onset_window_seconds=float(variant.get('ghost_onset_window_seconds', 0.05)),
            local_amp_ratio=float(variant.get('ghost_local_amp_ratio', 0.35)),
        )

    if variant.get('harmonic_shadow_prune', False):
        notes = remove_harmonic_shadow_notes(
            notes,
            onset_window_seconds=float(variant.get('harmonic_onset_window_seconds', 0.045)),
            shadow_amp_ratio=float(variant.get('harmonic_shadow_amp_ratio', 0.45)),
            intervals=variant.get('harmonic_intervals', (12, 19, 24)),
        )

    if variant.get('isolated_low_conf_prune', False):
        notes = remove_isolated_low_confidence_notes(
            notes,
            context_window_seconds=float(variant.get('isolated_context_window_seconds', 0.25)),
            isolated_amp_threshold=float(variant.get('isolated_amp_threshold', 0.38)),
        )

    if variant.get('cap_polyphony', False):
        notes = cap_polyphony_by_onset(
            notes,
            window_seconds=float(variant.get('polyphony_window_seconds', 0.04)),
            max_notes=int(variant.get('max_polyphony', 6)),
        )

    return sorted(notes, key=lambda n: (n['start'], n['midi']))


def run_basic_pitch_variant(audio_path, variant, use_cache=True):
    audio_for_bp = preprocess_audio_for_variant(audio_path, variant, use_cache=use_cache)
    raw_key = f"{_safe_audio_stem(audio_path)}__{variant['name']}__{_variant_hash(variant)}"
    raw_events = raw_basic_pitch_events(audio_for_bp, variant=variant, use_cache=use_cache, cache_key=raw_key)
    notes = postprocess_basic_pitch_events(raw_events, variant)
    return notes


In [ ]:

# -------------------------
# EXPERIMENT VARIANTS - V2
# -------------------------
# V1 result to beat on held-out test: baseline_amp030 F1 around 77.6%, higher_threshold_amp040 F1 around 78.6%.
# V2 adds methods aimed at getting back recall without exploding false positives.

BASE_VARIANTS = [
    {'name': 'baseline_amp030', 'amplitude_threshold': 0.30, 'min_midi': 40, 'max_midi': 88},
    {'name': 'prev_best_amp040', 'amplitude_threshold': 0.40, 'min_midi': 40, 'max_midi': 88},
]

# Basic Pitch internal threshold experiments.
# These change the note extraction stage before our amplitude/range filtering.
BP_PARAM_VARIANTS = [
    {'name': 'bp_onset040_frame030_amp030', 'amplitude_threshold': 0.30, 'min_midi': 40, 'max_midi': 88,
     'bp': {'onset_threshold': 0.40, 'frame_threshold': 0.30}},
    {'name': 'bp_onset060_frame030_amp030', 'amplitude_threshold': 0.30, 'min_midi': 40, 'max_midi': 88,
     'bp': {'onset_threshold': 0.60, 'frame_threshold': 0.30}},
    {'name': 'bp_onset050_frame025_amp035', 'amplitude_threshold': 0.35, 'min_midi': 40, 'max_midi': 88,
     'bp': {'onset_threshold': 0.50, 'frame_threshold': 0.25}},
    {'name': 'bp_onset050_frame040_amp030', 'amplitude_threshold': 0.30, 'min_midi': 40, 'max_midi': 88,
     'bp': {'onset_threshold': 0.50, 'frame_threshold': 0.40}},
    {'name': 'bp_min_note_30ms_amp030', 'amplitude_threshold': 0.30, 'min_midi': 40, 'max_midi': 88,
     'bp': {'minimum_note_length': 30}},
    {'name': 'bp_min_note_50ms_amp030', 'amplitude_threshold': 0.30, 'min_midi': 40, 'max_midi': 88,
     'bp': {'minimum_note_length': 50}},
]

# More granular amplitude threshold sweep near the known best region.
AMP_SWEEP_VARIANTS = [
    {'name': 'amp035', 'amplitude_threshold': 0.35, 'min_midi': 40, 'max_midi': 88},
    {'name': 'amp045', 'amplitude_threshold': 0.45, 'min_midi': 40, 'max_midi': 88},
    {'name': 'amp050', 'amplitude_threshold': 0.50, 'min_midi': 40, 'max_midi': 88},
]

# Adaptive thresholds per recording. Helpful if some recordings are quiet/noisy and one global threshold is suboptimal.
ADAPTIVE_VARIANTS = [
    {'name': 'adaptive_q20_floor025_ceil045', 'adaptive_amplitude': True, 'adaptive_quantile': 0.20, 'adaptive_floor': 0.25, 'adaptive_ceiling': 0.45, 'min_midi': 40, 'max_midi': 88},
    {'name': 'adaptive_q30_floor025_ceil045', 'adaptive_amplitude': True, 'adaptive_quantile': 0.30, 'adaptive_floor': 0.25, 'adaptive_ceiling': 0.45, 'min_midi': 40, 'max_midi': 88},
    {'name': 'adaptive_q40_floor020_ceil050', 'adaptive_amplitude': True, 'adaptive_quantile': 0.40, 'adaptive_floor': 0.20, 'adaptive_ceiling': 0.50, 'min_midi': 40, 'max_midi': 88},
]

# Safer postprocessing than v1's aggressive same-pitch merge.
POSTPROCESS_VARIANTS = [
    {'name': 'amp035_min_duration_30ms', 'amplitude_threshold': 0.35, 'min_midi': 40, 'max_midi': 88, 'min_duration': 0.03},
    {'name': 'amp040_min_duration_30ms', 'amplitude_threshold': 0.40, 'min_midi': 40, 'max_midi': 88, 'min_duration': 0.03},
    {'name': 'amp035_dedupe_same_pitch_25ms', 'amplitude_threshold': 0.35, 'min_midi': 40, 'max_midi': 88, 'dedupe_same_pitch_close': True, 'dedupe_window_seconds': 0.025},
    {'name': 'amp040_dedupe_same_pitch_25ms', 'amplitude_threshold': 0.40, 'min_midi': 40, 'max_midi': 88, 'dedupe_same_pitch_close': True, 'dedupe_window_seconds': 0.025},
    {'name': 'polyphony_cap6_amp030', 'amplitude_threshold': 0.30, 'min_midi': 40, 'max_midi': 88, 'cap_polyphony': True, 'max_polyphony': 6, 'polyphony_window_seconds': 0.04},
    {'name': 'polyphony_cap4_amp030', 'amplitude_threshold': 0.30, 'min_midi': 40, 'max_midi': 88, 'cap_polyphony': True, 'max_polyphony': 4, 'polyphony_window_seconds': 0.04},
]

# Extra preprocessing variants. V1 showed generic bandpass did not help globally, so these are exploratory.
PREPROCESS_VARIANTS = [
    {'name': 'lowpass_4200_amp030', 'amplitude_threshold': 0.30, 'min_midi': 40, 'max_midi': 88,
     'preprocess': {'sr': 22050, 'high_hz': 4200, 'filter_order': 4, 'normalize_peak': True}},
    {'name': 'bandpass_60_6000_amp030', 'amplitude_threshold': 0.30, 'min_midi': 40, 'max_midi': 88,
     'preprocess': {'sr': 22050, 'low_hz': 60, 'high_hz': 6000, 'filter_order': 4, 'normalize_peak': True}},
    {'name': 'bandpass_90_3500_amp030', 'amplitude_threshold': 0.30, 'min_midi': 40, 'max_midi': 88,
     'preprocess': {'sr': 22050, 'low_hz': 90, 'high_hz': 3500, 'filter_order': 4, 'normalize_peak': True}},
    {'name': 'hpss_mix_amp030', 'amplitude_threshold': 0.30, 'min_midi': 40, 'max_midi': 88,
     'preprocess': {'sr': 22050, 'hpss_mix': True, 'harmonic_weight': 1.0, 'percussive_weight': 0.25, 'normalize_peak': True}},
    {'name': 'rms_norm_amp035', 'amplitude_threshold': 0.35, 'min_midi': 40, 'max_midi': 88,
     'preprocess': {'sr': 22050, 'normalize_rms': True, 'target_rms': 0.07, 'normalize_peak': True}},
]

# Combo candidates that are plausible based on v1: cleaner threshold plus light duration/de-dupe/BP frame tuning.
COMBO_VARIANTS = [
    {'name': 'combo_amp040_min30_dedupe25', 'amplitude_threshold': 0.40, 'min_midi': 40, 'max_midi': 88, 'min_duration': 0.03, 'dedupe_same_pitch_close': True, 'dedupe_window_seconds': 0.025},
    {'name': 'combo_bp_frame040_amp035_min30', 'amplitude_threshold': 0.35, 'min_midi': 40, 'max_midi': 88, 'min_duration': 0.03, 'bp': {'onset_threshold': 0.50, 'frame_threshold': 0.40}},
    {'name': 'combo_adaptive_q30_dedupe25', 'adaptive_amplitude': True, 'adaptive_quantile': 0.30, 'adaptive_floor': 0.25, 'adaptive_ceiling': 0.45, 'min_midi': 40, 'max_midi': 88, 'dedupe_same_pitch_close': True, 'dedupe_window_seconds': 0.025},
]


# Dedicated ghost-note / artifact removal variants.
# These are designed to reduce false positives from faint accidental notes without using aggressive same-pitch merging.
GHOST_VARIANTS = [
    {'name': 'ghost_local_amp030_ratio035', 'amplitude_threshold': 0.30, 'min_midi': 40, 'max_midi': 88,
     'ghost_local_prune': True, 'ghost_local_amp_ratio': 0.35, 'ghost_onset_window_seconds': 0.05},
    {'name': 'ghost_local_amp035_ratio040', 'amplitude_threshold': 0.35, 'min_midi': 40, 'max_midi': 88,
     'ghost_local_prune': True, 'ghost_local_amp_ratio': 0.40, 'ghost_onset_window_seconds': 0.05},
    {'name': 'ghost_local_amp040_ratio035', 'amplitude_threshold': 0.40, 'min_midi': 40, 'max_midi': 88,
     'ghost_local_prune': True, 'ghost_local_amp_ratio': 0.35, 'ghost_onset_window_seconds': 0.05},

    {'name': 'ghost_harmonic_amp030_ratio045', 'amplitude_threshold': 0.30, 'min_midi': 40, 'max_midi': 88,
     'harmonic_shadow_prune': True, 'harmonic_shadow_amp_ratio': 0.45, 'harmonic_onset_window_seconds': 0.045,
     'harmonic_intervals': (12, 19, 24)},
    {'name': 'ghost_harmonic_amp035_ratio050', 'amplitude_threshold': 0.35, 'min_midi': 40, 'max_midi': 88,
     'harmonic_shadow_prune': True, 'harmonic_shadow_amp_ratio': 0.50, 'harmonic_onset_window_seconds': 0.045,
     'harmonic_intervals': (12, 19, 24)},

    {'name': 'ghost_isolated_low_conf_amp030', 'amplitude_threshold': 0.30, 'min_midi': 40, 'max_midi': 88,
     'isolated_low_conf_prune': True, 'isolated_amp_threshold': 0.38, 'isolated_context_window_seconds': 0.25},

    {'name': 'ghost_local_plus_harmonic_amp030', 'amplitude_threshold': 0.30, 'min_midi': 40, 'max_midi': 88,
     'ghost_local_prune': True, 'ghost_local_amp_ratio': 0.35, 'ghost_onset_window_seconds': 0.05,
     'harmonic_shadow_prune': True, 'harmonic_shadow_amp_ratio': 0.45, 'harmonic_onset_window_seconds': 0.045,
     'harmonic_intervals': (12, 19, 24)},

    {'name': 'combo_ghost_amp040_local_harmonic_dedupe', 'amplitude_threshold': 0.40, 'min_midi': 40, 'max_midi': 88,
     'ghost_local_prune': True, 'ghost_local_amp_ratio': 0.35, 'ghost_onset_window_seconds': 0.05,
     'harmonic_shadow_prune': True, 'harmonic_shadow_amp_ratio': 0.45, 'harmonic_onset_window_seconds': 0.045,
     'harmonic_intervals': (12, 19, 24),
     'dedupe_same_pitch_close': True, 'dedupe_window_seconds': 0.025},

    {'name': 'combo_ghost_amp035_local_harmonic_cap6', 'amplitude_threshold': 0.35, 'min_midi': 40, 'max_midi': 88,
     'ghost_local_prune': True, 'ghost_local_amp_ratio': 0.35, 'ghost_onset_window_seconds': 0.05,
     'harmonic_shadow_prune': True, 'harmonic_shadow_amp_ratio': 0.45, 'harmonic_onset_window_seconds': 0.045,
     'harmonic_intervals': (12, 19, 24),
     'cap_polyphony': True, 'max_polyphony': 6, 'polyphony_window_seconds': 0.04},
]

THRESHOLD_SWEEP_VARIANTS = [
    {'name': f'threshold_{str(t).replace(".", "p")}', 'amplitude_threshold': float(t), 'min_midi': 40, 'max_midi': 88}
    for t in [0.20, 0.25, 0.30, 0.325, 0.35, 0.375, 0.40, 0.425, 0.45, 0.475, 0.50]
]

if VARIANT_SET == 'core':
    EXPERIMENT_VARIANTS = BASE_VARIANTS + BP_PARAM_VARIANTS + AMP_SWEEP_VARIANTS + ADAPTIVE_VARIANTS + POSTPROCESS_VARIANTS + GHOST_VARIANTS + COMBO_VARIANTS
elif VARIANT_SET == 'full':
    EXPERIMENT_VARIANTS = BASE_VARIANTS + BP_PARAM_VARIANTS + AMP_SWEEP_VARIANTS + ADAPTIVE_VARIANTS + POSTPROCESS_VARIANTS + GHOST_VARIANTS + PREPROCESS_VARIANTS + COMBO_VARIANTS
elif VARIANT_SET == 'threshold_sweep':
    EXPERIMENT_VARIANTS = BASE_VARIANTS + THRESHOLD_SWEEP_VARIANTS
else:
    raise ValueError("VARIANT_SET must be one of: 'core', 'full', 'threshold_sweep'")

# De-duplicate by variant name in case a list overlaps.
seen = set()
unique = []
for v in EXPERIMENT_VARIANTS:
    if v['name'] not in seen:
        unique.append(v)
        seen.add(v['name'])
EXPERIMENT_VARIANTS = unique

print(f'Variant set: {VARIANT_SET}')
print(f'Variants to run: {len(EXPERIMENT_VARIANTS)}')
for v in EXPERIMENT_VARIANTS:
    bp = v.get('bp', {})
    pp = {k: v[k] for k in ['amplitude_threshold', 'adaptive_amplitude', 'min_duration', 'dedupe_same_pitch_close', 'cap_polyphony', 'ghost_local_prune', 'harmonic_shadow_prune', 'isolated_low_conf_prune'] if k in v}
    print(' -', v['name'], '| bp:', bp, '| post:', pp, '| preprocess:', bool(v.get('preprocess')))


Variant set: core
Variants to run: 32
 - baseline_amp030 | bp: {} | post: {'amplitude_threshold': 0.3} | preprocess: False
 - prev_best_amp040 | bp: {} | post: {'amplitude_threshold': 0.4} | preprocess: False
 - bp_onset040_frame030_amp030 | bp: {'onset_threshold': 0.4, 'frame_threshold': 0.3} | post: {'amplitude_threshold': 0.3} | preprocess: False
 - bp_onset060_frame030_amp030 | bp: {'onset_threshold': 0.6, 'frame_threshold': 0.3} | post: {'amplitude_threshold': 0.3} | preprocess: False
 - bp_onset050_frame025_amp035 | bp: {'onset_threshold': 0.5, 'frame_threshold': 0.25} | post: {'amplitude_threshold': 0.35} | preprocess: False
 - bp_onset050_frame040_amp030 | bp: {'onset_threshold': 0.5, 'frame_threshold': 0.4} | post: {'amplitude_threshold': 0.3} | preprocess: False
 - bp_min_note_30ms_amp030 | bp: {'minimum_note_length': 30} | post: {'amplitude_threshold': 0.3} | preprocess: False
 - bp_min_note_50ms_amp030 | bp: {'minimum_note_length': 50} | post: {'amplitude_threshold': 0.3} |

## 5b. V3 additions: two-pass detection, cluster pruning, and adaptive modes

These cells extend V2 without changing the working path setup. V2 showed that `amp040` improved precision but lost recall. V3 tries to recover recall while preventing extra false notes from making the tab messy.

Key ideas:

- **Two-pass Basic Pitch**: keep a clean high-threshold pass, then add plausible notes from a more recall-heavy pass.
- **Onset-cluster pruning**: group near-simultaneous notes and keep the strongest/playable subset.
- **Recording adaptive thresholding**: use different thresholds for solo vs comp or dense vs sparse predictions.
- **Timing diagnostics**: later cells sweep onset tolerance so we can tell if errors are note-detection errors or timing-offset errors.


In [ ]:

# -------------------------
# V3 HELPERS
# -------------------------
# Keep a reference to the V2 postprocessor so V3 can build on it safely.
_V2_POSTPROCESS_BASIC_PITCH_EVENTS = postprocess_basic_pitch_events
_V2_RUN_BASIC_PITCH_VARIANT = run_basic_pitch_variant


def _note_end(n):
    return float(n.get('end', float(n['start']) + float(n.get('duration', 0.0) or 0.0)))


def cluster_notes_by_onset(notes, window_seconds=0.05):
    """Group notes whose start times are close enough to act like one onset/chord cluster."""
    if not notes:
        return []
    notes_sorted = sorted([dict(n) for n in notes], key=lambda n: (float(n['start']), int(n['midi'])))
    clusters = []
    cluster = [notes_sorted[0]]
    cluster_start = float(notes_sorted[0]['start'])
    for n in notes_sorted[1:]:
        if float(n['start']) - cluster_start <= window_seconds:
            cluster.append(n)
        else:
            clusters.append(cluster)
            cluster = [n]
            cluster_start = float(n['start'])
    clusters.append(cluster)
    return clusters


def prune_onset_clusters(
    notes,
    window_seconds=0.05,
    max_notes=6,
    local_amp_ratio=0.25,
    prefer_guitar_range=True,
    remove_harmonic_shadows=True,
    harmonic_intervals=(12, 19, 24),
    harmonic_amp_ratio=0.45,
):
    """Plausibility filter for same-onset clusters.

    This is a more musical version of ghost-note removal:
    - group notes that start together
    - drop very weak notes relative to the strongest note in that onset cluster
    - optionally drop weak octave/fifth harmonic shadows
    - cap the cluster at max_notes because guitar has at most 6 strings
    """
    if not notes:
        return []

    kept = []
    for cluster in cluster_notes_by_onset(notes, window_seconds=window_seconds):
        if not cluster:
            continue
        max_amp = max(float(n.get('amplitude', 0.0)) for n in cluster)
        if max_amp <= 1e-9:
            candidate = list(cluster)
        else:
            candidate = [n for n in cluster if float(n.get('amplitude', 0.0)) >= local_amp_ratio * max_amp]

        if remove_harmonic_shadows and len(candidate) > 1:
            drop = set()
            for i, n in enumerate(candidate):
                for j, other in enumerate(candidate):
                    if i == j:
                        continue
                    interval = abs(int(n['midi']) - int(other['midi']))
                    weaker = float(n.get('amplitude', 0.0)) < harmonic_amp_ratio * float(other.get('amplitude', 0.0))
                    if interval in harmonic_intervals and weaker:
                        drop.add(i)
            candidate = [n for i, n in enumerate(candidate) if i not in drop]

        def score(n):
            amp = float(n.get('amplitude', 0.0))
            midi = int(n['midi'])
            guitar_bonus = 0.05 if (DEFAULT_MIN_MIDI <= midi <= DEFAULT_MAX_MIDI) else -0.20
            duration_bonus = min(float(n.get('duration', 0.0) or 0.0), 0.20) * 0.05
            return amp + (guitar_bonus if prefer_guitar_range else 0.0) + duration_bonus

        candidate = sorted(candidate, key=score, reverse=True)[:int(max_notes)]
        kept.extend(candidate)

    return sorted(kept, key=lambda n: (float(n['start']), int(n['midi'])))


def is_near_duplicate(note, existing, pitch_window=0, onset_window_seconds=0.03):
    """Check if note duplicates an existing note by pitch and onset."""
    for e in existing:
        if abs(int(note['midi']) - int(e['midi'])) <= pitch_window and abs(float(note['start']) - float(e['start'])) <= onset_window_seconds:
            return True
    return False


def merge_two_pass_notes(clean_notes, recall_notes, variant):
    """Keep clean-pass notes, then add plausible recall-pass notes.

    This is meant to recover notes that amp040 missed, while avoiding a flood of weak false positives.
    """
    notes = [dict(n, source=str(n.get('source', 'clean_pass')) + '+clean') for n in clean_notes]
    duplicate_window = float(variant.get('two_pass_duplicate_window_seconds', 0.035))
    recall_min_amp = float(variant.get('two_pass_recall_min_amp', 0.30))
    require_context = bool(variant.get('two_pass_require_context', False))
    context_window = float(variant.get('two_pass_context_window_seconds', 0.12))

    for n in sorted(recall_notes, key=lambda x: (float(x['start']), int(x['midi']))):
        if float(n.get('amplitude', 0.0)) < recall_min_amp:
            continue
        if is_near_duplicate(n, notes, pitch_window=0, onset_window_seconds=duplicate_window):
            continue
        if require_context:
            # Optional conservative mode: add recall notes only near an existing musical event.
            near_any = any(abs(float(n['start']) - float(e['start'])) <= context_window for e in notes)
            if not near_any:
                continue
        notes.append(dict(n, source=str(n.get('source', 'recall_pass')) + '+recall'))

    if variant.get('onset_cluster_prune', True):
        notes = prune_onset_clusters(
            notes,
            window_seconds=float(variant.get('cluster_window_seconds', 0.05)),
            max_notes=int(variant.get('cluster_max_notes', 6)),
            local_amp_ratio=float(variant.get('cluster_local_amp_ratio', 0.25)),
            remove_harmonic_shadows=bool(variant.get('cluster_remove_harmonic_shadows', True)),
            harmonic_amp_ratio=float(variant.get('cluster_harmonic_amp_ratio', 0.45)),
        )

    if variant.get('dedupe_same_pitch_close', True):
        notes = drop_near_duplicate_same_pitch(notes, window_seconds=float(variant.get('dedupe_window_seconds', 0.025)))

    return sorted(notes, key=lambda n: (float(n['start']), int(n['midi'])))


def resolve_density_adaptive_variant(raw_events, variant, audio_path=None):
    """Choose a threshold based on raw note density and confidence distribution.

    This is intentionally simple and transparent. Dense/noisy files get a stricter threshold;
    sparse/quiet files get a slightly lower threshold to recover recall.
    """
    if not raw_events:
        return dict(variant, amplitude_threshold=float(variant.get('quiet_amp', 0.35)))

    duration = max(float(e.get('end', e.get('start', 0.0))) for e in raw_events) if raw_events else 0.0
    duration = max(duration, 1e-6)
    note_rate = len(raw_events) / duration
    amps = np.array([float(e.get('amplitude', 0.0)) for e in raw_events], dtype=float)
    med_amp = float(np.median(amps)) if len(amps) else 0.0

    density_cutoff = float(variant.get('density_note_rate_cutoff', 8.0))
    median_cutoff = float(variant.get('density_median_amp_cutoff', 0.34))
    noisy_amp = float(variant.get('noisy_amp', 0.40))
    quiet_amp = float(variant.get('quiet_amp', 0.35))
    very_quiet_amp = float(variant.get('very_quiet_amp', 0.325))

    chosen = noisy_amp if note_rate >= density_cutoff else quiet_amp
    if med_amp < median_cutoff and note_rate < density_cutoff:
        chosen = very_quiet_amp

    resolved = dict(variant)
    resolved['amplitude_threshold'] = chosen
    resolved['_density_note_rate'] = note_rate
    resolved['_density_median_amp'] = med_amp
    resolved['_density_chosen_amp'] = chosen
    return resolved


def resolve_style_adaptive_variant(variant, audio_path):
    """Choose a threshold/params by recording stem: solo favors recall, comp favors precision."""
    stem = Path(audio_path).stem
    resolved = dict(variant)
    if '_solo' in stem:
        resolved['amplitude_threshold'] = float(variant.get('solo_amp', 0.35))
        if 'solo_bp' in variant:
            resolved['bp'] = dict(variant['solo_bp'])
    elif '_comp' in stem:
        resolved['amplitude_threshold'] = float(variant.get('comp_amp', 0.40))
        if 'comp_bp' in variant:
            resolved['bp'] = dict(variant['comp_bp'])
    else:
        resolved['amplitude_threshold'] = float(variant.get('default_amp', 0.375))
    return resolved


def postprocess_basic_pitch_events(raw_events, variant):
    """V3 postprocess: run V2 cleanup, then optional onset-cluster plausibility pruning."""
    notes = _V2_POSTPROCESS_BASIC_PITCH_EVENTS(raw_events, variant)

    if variant.get('onset_cluster_prune', False):
        notes = prune_onset_clusters(
            notes,
            window_seconds=float(variant.get('cluster_window_seconds', 0.05)),
            max_notes=int(variant.get('cluster_max_notes', 6)),
            local_amp_ratio=float(variant.get('cluster_local_amp_ratio', 0.25)),
            remove_harmonic_shadows=bool(variant.get('cluster_remove_harmonic_shadows', True)),
            harmonic_amp_ratio=float(variant.get('cluster_harmonic_amp_ratio', 0.45)),
        )

    return sorted(notes, key=lambda n: (float(n['start']), int(n['midi'])))


def run_basic_pitch_variant(audio_path, variant, use_cache=True):
    """V3 runner with support for two-pass and adaptive variants."""
    audio_for_bp = preprocess_audio_for_variant(audio_path, variant, use_cache=use_cache)

    # Style-adaptive variant chooses params from filename before running Basic Pitch.
    if variant.get('style_adaptive', False):
        variant = resolve_style_adaptive_variant(variant, audio_path)

    # Two-pass ensemble: one clean pass plus one recall pass.
    if variant.get('two_pass_ensemble', False):
        clean_cfg = dict(variant.get('clean_pass', {}))
        recall_cfg = dict(variant.get('recall_pass', {}))
        clean_cfg.setdefault('name', variant['name'] + '__clean')
        recall_cfg.setdefault('name', variant['name'] + '__recall')
        clean_cfg.setdefault('min_midi', variant.get('min_midi', 40))
        clean_cfg.setdefault('max_midi', variant.get('max_midi', 88))
        recall_cfg.setdefault('min_midi', variant.get('min_midi', 40))
        recall_cfg.setdefault('max_midi', variant.get('max_midi', 88))

        raw_clean = raw_basic_pitch_events(
            audio_for_bp,
            variant=clean_cfg,
            use_cache=use_cache,
            cache_key=f"{_safe_audio_stem(audio_path)}__{variant['name']}__clean__{_variant_hash(clean_cfg)}",
        )
        raw_recall = raw_basic_pitch_events(
            audio_for_bp,
            variant=recall_cfg,
            use_cache=use_cache,
            cache_key=f"{_safe_audio_stem(audio_path)}__{variant['name']}__recall__{_variant_hash(recall_cfg)}",
        )
        clean_notes = _V2_POSTPROCESS_BASIC_PITCH_EVENTS(raw_clean, clean_cfg)
        recall_notes = _V2_POSTPROCESS_BASIC_PITCH_EVENTS(raw_recall, recall_cfg)
        return merge_two_pass_notes(clean_notes, recall_notes, variant)

    raw_key = f"{_safe_audio_stem(audio_path)}__{variant['name']}__{_variant_hash(variant)}"
    raw_events = raw_basic_pitch_events(audio_for_bp, variant=variant, use_cache=use_cache, cache_key=raw_key)

    if variant.get('density_adaptive', False):
        variant = resolve_density_adaptive_variant(raw_events, variant, audio_path=audio_path)

    return postprocess_basic_pitch_events(raw_events, variant)


# -------------------------
# V3 VARIANTS
# -------------------------
# These append to the V2 variant lists. They are designed to test whether we can beat amp040
# by recovering recall without making MIDI/tab output messy.
V3_BP_PARAM_VARIANTS = [
    {'name': 'v3_bp_onset045_frame025_amp0375', 'amplitude_threshold': 0.375, 'min_midi': 40, 'max_midi': 88,
     'bp': {'onset_threshold': 0.45, 'frame_threshold': 0.25}},
    {'name': 'v3_bp_onset055_frame025_amp035', 'amplitude_threshold': 0.35, 'min_midi': 40, 'max_midi': 88,
     'bp': {'onset_threshold': 0.55, 'frame_threshold': 0.25}},
    {'name': 'v3_bp_onset050_frame020_amp040', 'amplitude_threshold': 0.40, 'min_midi': 40, 'max_midi': 88,
     'bp': {'onset_threshold': 0.50, 'frame_threshold': 0.20}},
]

V3_CLUSTER_VARIANTS = [
    {'name': 'v3_cluster_amp030_ratio025_cap6', 'amplitude_threshold': 0.30, 'min_midi': 40, 'max_midi': 88,
     'onset_cluster_prune': True, 'cluster_window_seconds': 0.05, 'cluster_local_amp_ratio': 0.25, 'cluster_max_notes': 6,
     'cluster_remove_harmonic_shadows': True, 'cluster_harmonic_amp_ratio': 0.45},
    {'name': 'v3_cluster_amp035_ratio030_cap6', 'amplitude_threshold': 0.35, 'min_midi': 40, 'max_midi': 88,
     'onset_cluster_prune': True, 'cluster_window_seconds': 0.05, 'cluster_local_amp_ratio': 0.30, 'cluster_max_notes': 6,
     'cluster_remove_harmonic_shadows': True, 'cluster_harmonic_amp_ratio': 0.45},
    {'name': 'v3_cluster_amp035_ratio035_cap4', 'amplitude_threshold': 0.35, 'min_midi': 40, 'max_midi': 88,
     'onset_cluster_prune': True, 'cluster_window_seconds': 0.05, 'cluster_local_amp_ratio': 0.35, 'cluster_max_notes': 4,
     'cluster_remove_harmonic_shadows': True, 'cluster_harmonic_amp_ratio': 0.45},
]

V3_ADAPTIVE_VARIANTS = [
    {'name': 'v3_density_adaptive_amp0325_035_040', 'density_adaptive': True,
     'quiet_amp': 0.35, 'very_quiet_amp': 0.325, 'noisy_amp': 0.40,
     'density_note_rate_cutoff': 8.0, 'density_median_amp_cutoff': 0.34,
     'min_midi': 40, 'max_midi': 88},
    {'name': 'v3_style_adaptive_solo035_comp040', 'style_adaptive': True,
     'solo_amp': 0.35, 'comp_amp': 0.40, 'default_amp': 0.375,
     'solo_bp': {'onset_threshold': 0.50, 'frame_threshold': 0.25},
     'comp_bp': {'onset_threshold': 0.50, 'frame_threshold': 0.30},
     'min_midi': 40, 'max_midi': 88},
]

V3_TWO_PASS_VARIANTS = [
    {'name': 'v3_two_pass_amp040_plus_recall035_cluster', 'two_pass_ensemble': True, 'min_midi': 40, 'max_midi': 88,
     'clean_pass': {'amplitude_threshold': 0.40, 'min_midi': 40, 'max_midi': 88},
     'recall_pass': {'amplitude_threshold': 0.35, 'min_midi': 40, 'max_midi': 88, 'bp': {'onset_threshold': 0.50, 'frame_threshold': 0.25}},
     'two_pass_recall_min_amp': 0.35, 'two_pass_duplicate_window_seconds': 0.035,
     'onset_cluster_prune': True, 'cluster_window_seconds': 0.05, 'cluster_local_amp_ratio': 0.25, 'cluster_max_notes': 6},
    {'name': 'v3_two_pass_amp040_plus_recall030_strict_cluster', 'two_pass_ensemble': True, 'min_midi': 40, 'max_midi': 88,
     'clean_pass': {'amplitude_threshold': 0.40, 'min_midi': 40, 'max_midi': 88},
     'recall_pass': {'amplitude_threshold': 0.30, 'min_midi': 40, 'max_midi': 88, 'bp': {'onset_threshold': 0.50, 'frame_threshold': 0.25}},
     'two_pass_recall_min_amp': 0.30, 'two_pass_duplicate_window_seconds': 0.035,
     'onset_cluster_prune': True, 'cluster_window_seconds': 0.05, 'cluster_local_amp_ratio': 0.35, 'cluster_max_notes': 6},
    {'name': 'v3_two_pass_amp040_plus_recall035_context', 'two_pass_ensemble': True, 'min_midi': 40, 'max_midi': 88,
     'clean_pass': {'amplitude_threshold': 0.40, 'min_midi': 40, 'max_midi': 88},
     'recall_pass': {'amplitude_threshold': 0.35, 'min_midi': 40, 'max_midi': 88, 'bp': {'onset_threshold': 0.50, 'frame_threshold': 0.25}},
     'two_pass_recall_min_amp': 0.35, 'two_pass_duplicate_window_seconds': 0.035,
     'two_pass_require_context': True, 'two_pass_context_window_seconds': 0.12,
     'onset_cluster_prune': True, 'cluster_window_seconds': 0.05, 'cluster_local_amp_ratio': 0.25, 'cluster_max_notes': 6},
]

V3_VARIANTS = V3_BP_PARAM_VARIANTS + V3_CLUSTER_VARIANTS + V3_ADAPTIVE_VARIANTS + V3_TWO_PASS_VARIANTS

if VARIANT_SET in ['core', 'full']:
    EXPERIMENT_VARIANTS = EXPERIMENT_VARIANTS + V3_VARIANTS
elif VARIANT_SET == 'threshold_sweep':
    # leave threshold sweep clean and comparable
    pass

# De-duplicate again after appending V3 variants.
seen = set()
unique = []
for v in EXPERIMENT_VARIANTS:
    if v['name'] not in seen:
        unique.append(v)
        seen.add(v['name'])
EXPERIMENT_VARIANTS = unique

print('\nV3 enabled.')
print(f'Final variant count: {len(EXPERIMENT_VARIANTS)}')
print('V3 variants added:')
for v in V3_VARIANTS:
    print(' -', v['name'])



V3 enabled.
Final variant count: 43
V3 variants added:
 - v3_bp_onset045_frame025_amp0375
 - v3_bp_onset055_frame025_amp035
 - v3_bp_onset050_frame020_amp040
 - v3_cluster_amp030_ratio025_cap6
 - v3_cluster_amp035_ratio030_cap6
 - v3_cluster_amp035_ratio035_cap4
 - v3_density_adaptive_amp0325_035_040
 - v3_style_adaptive_solo035_comp040
 - v3_two_pass_amp040_plus_recall035_cluster
 - v3_two_pass_amp040_plus_recall030_strict_cluster
 - v3_two_pass_amp040_plus_recall035_context


## 6. Note-capture matching metrics

This evaluates Basic Pitch directly against GuitarSet notes before any fretboard assignment.

A predicted note is a match when:

```text
same MIDI pitch AND |predicted_onset - ground_truth_onset| <= AUDIO_MATCH_ONSET_TOLERANCE_SECONDS
```

Matching is one-to-one and greedy by smallest onset error.


In [ ]:
def valid_gt_notes(record):
    """Ground-truth notes with a valid MIDI pitch. String/fret are not needed here."""
    out = []
    for i, n in enumerate(record.get('notes', [])):
        if n.get('midi') is None:
            continue
        row = dict(n)
        row['_gt_idx'] = i
        out.append(row)
    return sorted(out, key=lambda n: (float(n['start']), int(n['midi'])))


def match_pred_notes_to_gt(pred_notes, gt_notes, onset_tolerance=AUDIO_MATCH_ONSET_TOLERANCE_SECONDS, require_pitch=True):
    pred = [dict(p, _pred_idx=i) for i, p in enumerate(pred_notes) if p.get('midi') is not None]
    gt = [dict(g, _gt_idx=i) for i, g in enumerate(gt_notes) if g.get('midi') is not None]

    candidates = []
    for pi, p in enumerate(pred):
        for gi, g in enumerate(gt):
            if require_pitch and int(p['midi']) != int(g['midi']):
                continue
            dt = abs(float(p['start']) - float(g['start']))
            if dt <= onset_tolerance:
                candidates.append((dt, pi, gi))

    candidates.sort(key=lambda x: x[0])
    used_p, used_g, matches = set(), set(), []
    for dt, pi, gi in candidates:
        if pi in used_p or gi in used_g:
            continue
        used_p.add(pi)
        used_g.add(gi)
        p = pred[pi]
        g = gt[gi]
        matches.append({
            'pred_idx': p['_pred_idx'],
            'gt_idx': g['_gt_idx'],
            'pred_start': float(p['start']),
            'gt_start': float(g['start']),
            'onset_error_sec': float(dt),
            'midi': int(p['midi']),
            'pred_duration': float(p.get('duration', 0.0) or 0.0),
            'gt_duration': float(g.get('duration', 0.0) or 0.0),
            'pred_amplitude': float(p.get('amplitude', np.nan)),
        })

    unmatched_pred = [p for i, p in enumerate(pred) if i not in used_p]
    unmatched_gt = [g for i, g in enumerate(gt) if i not in used_g]
    return matches, unmatched_pred, unmatched_gt


def evaluate_basic_pitch_variant_for_record(record, audio_path, variant, use_cache=True):
    t0 = pd.Timestamp.now()
    pred_notes = run_basic_pitch_variant(audio_path, variant, use_cache=use_cache)
    gt_notes = valid_gt_notes(record)

    matches, false_pos, missed = match_pred_notes_to_gt(pred_notes, gt_notes)

    n_pred = len(pred_notes)
    n_gt = len(gt_notes)
    n_match = len(matches)
    precision = n_match / n_pred if n_pred else 0.0
    recall = n_match / n_gt if n_gt else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    mean_onset_error = float(np.mean([m['onset_error_sec'] for m in matches])) if matches else np.nan
    median_onset_error = float(np.median([m['onset_error_sec'] for m in matches])) if matches else np.nan
    runtime_sec = (pd.Timestamp.now() - t0).total_seconds()

    metrics = {
        'recording': record['recording'],
        'variant': variant['name'],
        'audio_path': str(audio_path),
        'is_solo': record['recording'].endswith('_solo'),
        'is_comp': record['recording'].endswith('_comp'),
        'n_gt_notes': n_gt,
        'n_pred_notes': n_pred,
        'n_pitch_onset_matches': n_match,
        'n_false_positives': len(false_pos),
        'n_missed_gt_notes': len(missed),
        'pitch_precision': precision,
        'pitch_recall': recall,
        'pitch_f1': f1,
        'mean_onset_error_sec': mean_onset_error,
        'median_onset_error_sec': median_onset_error,
        'runtime_sec': runtime_sec,
        'amplitude_threshold': variant.get('amplitude_threshold', np.nan),
        'min_midi': variant.get('min_midi', np.nan),
        'max_midi': variant.get('max_midi', np.nan),
        'min_duration': variant.get('min_duration', 0.0),
        'merge_same_pitch': bool(variant.get('merge_same_pitch', False)),
        'preprocess': json.dumps(variant.get('preprocess', {}), sort_keys=True),
        'bp_params': json.dumps(variant.get('bp', {}), sort_keys=True),
        'adaptive_amplitude': bool(variant.get('adaptive_amplitude', False)),
        'dedupe_same_pitch_close': bool(variant.get('dedupe_same_pitch_close', False)),
        'cap_polyphony': bool(variant.get('cap_polyphony', False)),
        'ghost_local_prune': bool(variant.get('ghost_local_prune', False)),
        'ghost_local_amp_ratio': variant.get('ghost_local_amp_ratio', np.nan),
        'harmonic_shadow_prune': bool(variant.get('harmonic_shadow_prune', False)),
        'harmonic_shadow_amp_ratio': variant.get('harmonic_shadow_amp_ratio', np.nan),
        'isolated_low_conf_prune': bool(variant.get('isolated_low_conf_prune', False)),
        'isolated_amp_threshold': variant.get('isolated_amp_threshold', np.nan),
    }

    pred_df = pd.DataFrame(pred_notes)
    if not pred_df.empty:
        pred_df['recording'] = record['recording']
        pred_df['variant'] = variant['name']

    matches_df = pd.DataFrame(matches)
    if not matches_df.empty:
        matches_df['recording'] = record['recording']
        matches_df['variant'] = variant['name']

    false_pos_df = pd.DataFrame(false_pos)
    if not false_pos_df.empty:
        false_pos_df['recording'] = record['recording']
        false_pos_df['variant'] = variant['name']

    missed_df = pd.DataFrame(missed)
    if not missed_df.empty:
        missed_df['recording'] = record['recording']
        missed_df['variant'] = variant['name']

    return metrics, pred_df, matches_df, false_pos_df, missed_df



## 7. Run the experiment suite

Start with `MAX_AUDIO_RECORDINGS = 5`. Once it runs cleanly, change it to `None` in the config cell and rerun from the pairing cell onward.


In [ ]:
records_to_run = paired_records if MAX_AUDIO_RECORDINGS is None else paired_records[:MAX_AUDIO_RECORDINGS]
print(f'Running Basic Pitch note-capture experiments on {len(records_to_run)} recordings from {AUDIO_EVAL_LABEL}.')
print(f'Onset match tolerance: {AUDIO_MATCH_ONSET_TOLERANCE_SECONDS} sec')
print(f'Variants: {len(EXPERIMENT_VARIANTS)}')

all_metrics = []
all_predictions = []
all_matches = []
all_false_pos = []
all_missed = []
failed = []

for record, audio_path in records_to_run:
    print(f"\nRecording: {record['recording']} | audio: {audio_path.name}")
    for variant in EXPERIMENT_VARIANTS:
        print(f"  - {variant['name']}")
        try:
            metrics, pred_df, matches_df, false_pos_df, missed_df = evaluate_basic_pitch_variant_for_record(
                record, audio_path, variant, use_cache=True
            )
            all_metrics.append(metrics)
            if not pred_df.empty:
                all_predictions.append(pred_df)
            if not matches_df.empty:
                all_matches.append(matches_df)
            if not false_pos_df.empty:
                all_false_pos.append(false_pos_df)
            if not missed_df.empty:
                all_missed.append(missed_df)
        except Exception as e:
            print('    FAILED:', repr(e))
            failed.append({
                'recording': record['recording'],
                'variant': variant['name'],
                'audio_path': str(audio_path),
                'error': repr(e),
            })

metrics_df = pd.DataFrame(all_metrics)
predictions_df = pd.concat(all_predictions, ignore_index=True) if all_predictions else pd.DataFrame()
matches_df = pd.concat(all_matches, ignore_index=True) if all_matches else pd.DataFrame()
false_pos_df = pd.concat(all_false_pos, ignore_index=True) if all_false_pos else pd.DataFrame()
missed_df = pd.concat(all_missed, ignore_index=True) if all_missed else pd.DataFrame()
failed_df = pd.DataFrame(failed)

# Save detailed outputs.
metrics_path = OUTPUT_DIR / 'basic_pitch_variant_metrics_by_record.csv'
pred_path = OUTPUT_DIR / 'basic_pitch_predictions.csv'
match_path = OUTPUT_DIR / 'basic_pitch_pitch_onset_matches.csv'
fp_path = OUTPUT_DIR / 'basic_pitch_false_positives.csv'
miss_path = OUTPUT_DIR / 'basic_pitch_missed_gt_notes.csv'
failed_path = OUTPUT_DIR / 'basic_pitch_failed_runs.csv'

metrics_df.to_csv(metrics_path, index=False)
predictions_df.to_csv(pred_path, index=False)
matches_df.to_csv(match_path, index=False)
false_pos_df.to_csv(fp_path, index=False)
missed_df.to_csv(miss_path, index=False)
failed_df.to_csv(failed_path, index=False)

print('\nSaved:')
for p in [metrics_path, pred_path, match_path, fp_path, miss_path, failed_path]:
    print(' -', p.resolve())

if not failed_df.empty:
    print('\nFailed runs:')
    display(failed_df)



Streaming output truncated to the last 5000 lines.
  - ghost_isolated_low_conf_amp030
Running Basic Pitch on: 00_Rock3-117-Bb_solo_mic.wav | bp_kwargs={}
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/00_Rock3-117-Bb_solo_mic.wav...
  - ghost_local_plus_harmonic_amp030
Running Basic Pitch on: 00_Rock3-117-Bb_solo_mic.wav | bp_kwargs={}
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/00_Rock3-117-Bb_solo_mic.wav...
  - combo_ghost_amp040_local_harmonic_dedupe
Running Basic Pitch on: 00_Rock3-117-Bb_solo_mic.wav | bp_kwargs={}
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/00_Rock3-117-Bb_solo_mic.wav...
  - combo_ghost_amp035_local_harmonic_cap6
Running Basic Pitch on: 00_Rock3-117-Bb_solo_mic.wav | bp_kwargs={}
Predicting MIDI for /content/drive/MyDrive/Capstone/FullGuitarSetData/AudioFiles/00_Rock3-117-Bb_solo_mic.wav...
  - combo_amp040_min30_dedupe25
Running Basic Pitch on: 00_Rock3-

## 8. Summary tables

The most important table is the weighted summary by variant. It uses total notes across recordings so large recordings count proportionally.


In [ ]:
def summarize_variant_metrics(metrics_df):
    if metrics_df.empty:
        return pd.DataFrame()
    rows = []
    for variant, g in metrics_df.groupby('variant'):
        n_pred = int(g['n_pred_notes'].sum())
        n_gt = int(g['n_gt_notes'].sum())
        n_match = int(g['n_pitch_onset_matches'].sum())
        precision = n_match / n_pred if n_pred else 0.0
        recall = n_match / n_gt if n_gt else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        rows.append({
            'variant': variant,
            'recordings': int(g['recording'].nunique()),
            'n_gt_notes': n_gt,
            'n_pred_notes': n_pred,
            'n_pitch_onset_matches': n_match,
            'n_false_positives': int(g['n_false_positives'].sum()),
            'n_missed_gt_notes': int(g['n_missed_gt_notes'].sum()),
            'pitch_precision': precision,
            'pitch_recall': recall,
            'pitch_f1': f1,
            'mean_onset_error_sec': float(np.average(g['mean_onset_error_sec'].fillna(0), weights=np.maximum(g['n_pitch_onset_matches'], 1))) if len(g) else np.nan,
            'mean_runtime_sec_per_recording': float(g['runtime_sec'].mean()),
        })
    out = pd.DataFrame(rows).sort_values(['pitch_f1', 'pitch_recall', 'pitch_precision'], ascending=False)
    return out

summary_df = summarize_variant_metrics(metrics_df)
summary_path = OUTPUT_DIR / 'basic_pitch_variant_summary_weighted.csv'
summary_df.to_csv(summary_path, index=False)

print('Saved summary to:', summary_path.resolve())
display(summary_df.round(4))

if not summary_df.empty:
    best = summary_df.iloc[0]
    print('\nBest by pitch_f1:')
    print(best[['variant', 'pitch_precision', 'pitch_recall', 'pitch_f1', 'n_pred_notes', 'n_gt_notes', 'n_false_positives', 'n_missed_gt_notes']])



Saved summary to: /content/drive/.shortcut-targets-by-id/1JNqe8bukG93wCWVxbk7SKlNvVyIZHyTC/Capstone/outputs/basic_pitch_note_capture_experiments_v3/basic_pitch_variant_summary_weighted.csv


,variant,recordings,n_gt_notes,n_pred_notes,n_pitch_onset_matches,n_false_positives,n_missed_gt_notes,pitch_precision,pitch_recall,pitch_f1,mean_onset_error_sec,mean_runtime_sec_per_recording
33,v3_bp_onset050_frame020_amp040,54,10207,8586,7392,1194,2815,0.8609,0.7242,0.7867,0.0112,1.6086
34,v3_bp_onset055_frame025_amp035,54,10207,9046,7571,1475,2636,0.8369,0.7417,0.7865,0.0113,1.5500
41,v3_two_pass_amp040_plus_recall035_cluster,54,10207,9437,7721,1716,2486,0.8182,0.7564,0.7861,0.0116,3.0914
6,amp040_dedupe_same_pitch_25ms,54,10207,8765,7456,1309,2751,0.8507,0.7305,0.7860,0.0114,1.5041
7,amp040_min_duration_30ms,54,10207,8765,7456,1309,2751,0.8507,0.7305,0.7860,0.0114,1.5523
18,combo_amp040_min30_dedupe25,54,10207,8765,7456,1309,2751,0.8507,0.7305,0.7860,0.0114,1.5013
21,combo_ghost_amp040_local_harmonic_dedupe,54,10207,8765,7456,1309,2751,0.8507,0.7305,0.7860,0.0114,1.5997
27,ghost_local_amp040_ratio035,54,10207,8765,7456,1309,2751,0.8507,0.7305,0.7860,0.0114,1.6108
31,prev_best_amp040,54,10207,8765,7456,1309,2751,0.8507,0.7305,0.7860,0.0114,1.5306
14,bp_onset050_frame025_amp035,54,10207,9345,7683,1662,2524,0.8222,0.7527,0.7859,0.0114,1.5763



Best by pitch_f1:
variant              v3_bp_onset050_frame020_amp040
pitch_precision                            0.860936
pitch_recall                               0.724209
pitch_f1                                   0.786676
n_pred_notes                                   8586
n_gt_notes                                    10207
n_false_positives                              1194
n_missed_gt_notes                              2815
Name: 33, dtype: object


## 8b. V3 onset-tolerance diagnostic

This does **not** rerun Basic Pitch. It re-scores the saved predictions with wider onset tolerances to answer: are the misses true note-detection misses, or is Basic Pitch finding the right pitch slightly early/late?


In [ ]:

def recompute_metrics_from_predictions_for_tolerance(predictions_df, records_to_run, variants, tolerance_seconds):
    if predictions_df.empty:
        return pd.DataFrame()

    record_lookup = {r['recording']: r for r, _ in records_to_run}
    rows = []
    for variant in variants:
        vname = variant['name']
        pred_v = predictions_df[predictions_df['variant'] == vname]
        if pred_v.empty:
            continue

        total_gt = total_pred = total_match = 0
        for recording, pred_g in pred_v.groupby('recording'):
            record = record_lookup.get(recording)
            if record is None:
                continue
            gt_notes = valid_gt_notes(record)
            pred_notes = pred_g.to_dict('records')
            matches, fp, missed = match_pred_notes_to_gt(pred_notes, gt_notes, onset_tolerance=tolerance_seconds)
            total_gt += len(gt_notes)
            total_pred += len(pred_notes)
            total_match += len(matches)

        precision = total_match / total_pred if total_pred else 0.0
        recall = total_match / total_gt if total_gt else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append({
            'variant': vname,
            'onset_tolerance_sec': tolerance_seconds,
            'n_gt_notes': total_gt,
            'n_pred_notes': total_pred,
            'n_pitch_onset_matches': total_match,
            'pitch_precision': precision,
            'pitch_recall': recall,
            'pitch_f1': f1,
            'n_false_positives': total_pred - total_match,
            'n_missed_gt_notes': total_gt - total_match,
        })
    return pd.DataFrame(rows)

ONSET_TOLERANCE_SWEEP = [0.05, 0.075, 0.10, 0.125]
tol_tables = []
for tol in ONSET_TOLERANCE_SWEEP:
    tol_tables.append(recompute_metrics_from_predictions_for_tolerance(predictions_df, records_to_run, EXPERIMENT_VARIANTS, tol))

tolerance_df = pd.concat(tol_tables, ignore_index=True) if tol_tables else pd.DataFrame()
tolerance_path = OUTPUT_DIR / 'basic_pitch_onset_tolerance_sweep.csv'
tolerance_df.to_csv(tolerance_path, index=False)
print('Saved onset tolerance sweep to:', tolerance_path.resolve())

if not tolerance_df.empty:
    display(
        tolerance_df.sort_values(['onset_tolerance_sec', 'pitch_f1'], ascending=[True, False])
        .groupby('onset_tolerance_sec')
        .head(8)
        .round(4)
    )


## 8c. V3 validation/test discipline helper

Recommended workflow:

1. Set `USE_HELDOUT_SPLIT = False` only for exploratory all-record diagnostics, not final claims.
2. For defensible tuning, temporarily evaluate variants on `VAL_RECORDS`, pick the top few, then report final results on `TEST_RECORDS`.
3. If you already ran test, use the result as exploratory and avoid claiming that V3 was tuned without seeing test performance.

The helper below saves the top variants from the current run so you can rerun only those in a follow-up notebook/config.


In [ ]:

TOP_K_VARIANTS_TO_RERUN = 6
if not summary_df.empty:
    top_variant_names = summary_df.head(TOP_K_VARIANTS_TO_RERUN)['variant'].tolist()
    top_variants_path = OUTPUT_DIR / 'top_variants_to_rerun.json'
    with open(top_variants_path, 'w') as f:
        json.dump(top_variant_names, f, indent=2)
    print('Top variants from this run:')
    for name in top_variant_names:
        print(' -', name)
    print('Saved to:', top_variants_path.resolve())


Top variants from this run:
 - v3_bp_onset050_frame020_amp040
 - v3_bp_onset055_frame025_amp035
 - v3_two_pass_amp040_plus_recall035_cluster
 - amp040_dedupe_same_pitch_25ms
 - amp040_min_duration_30ms
 - combo_amp040_min30_dedupe25
Saved to: /content/drive/.shortcut-targets-by-id/1JNqe8bukG93wCWVxbk7SKlNvVyIZHyTC/Capstone/outputs/basic_pitch_note_capture_experiments_v3/top_variants_to_rerun.json


In [ ]:
# Compare every variant against the baseline.
BASELINE_VARIANT = 'baseline_amp030'
if not summary_df.empty and BASELINE_VARIANT in set(summary_df['variant']):
    base = summary_df[summary_df['variant'] == BASELINE_VARIANT].iloc[0]
    comparison_df = summary_df.copy()
    for col in ['pitch_precision', 'pitch_recall', 'pitch_f1', 'n_pred_notes', 'n_false_positives', 'n_missed_gt_notes']:
        comparison_df[f'delta_{col}_vs_baseline'] = comparison_df[col] - base[col]
    comparison_path = OUTPUT_DIR / 'basic_pitch_variant_comparison_vs_baseline.csv'
    comparison_df.to_csv(comparison_path, index=False)
    print('Saved baseline comparison to:', comparison_path.resolve())
    display(comparison_df.round(4))
else:
    print(f'Baseline variant {BASELINE_VARIANT!r} not found in summary_df.')



Saved baseline comparison to: /content/drive/.shortcut-targets-by-id/1JNqe8bukG93wCWVxbk7SKlNvVyIZHyTC/Capstone/outputs/basic_pitch_note_capture_experiments_v3/basic_pitch_variant_comparison_vs_baseline.csv


,variant,recordings,n_gt_notes,n_pred_notes,n_pitch_onset_matches,n_false_positives,n_missed_gt_notes,pitch_precision,pitch_recall,pitch_f1,mean_onset_error_sec,mean_runtime_sec_per_recording,delta_pitch_precision_vs_baseline,delta_pitch_recall_vs_baseline,delta_pitch_f1_vs_baseline,delta_n_pred_notes_vs_baseline,delta_n_false_positives_vs_baseline,delta_n_missed_gt_notes_vs_baseline
33,v3_bp_onset050_frame020_amp040,54,10207,8586,7392,1194,2815,0.8609,0.7242,0.7867,0.0112,1.6086,0.0588,-0.0270,0.0108,-974,-698,276
34,v3_bp_onset055_frame025_amp035,54,10207,9046,7571,1475,2636,0.8369,0.7417,0.7865,0.0113,1.5500,0.0349,-0.0095,0.0106,-514,-417,97
41,v3_two_pass_amp040_plus_recall035_cluster,54,10207,9437,7721,1716,2486,0.8182,0.7564,0.7861,0.0116,3.0914,0.0161,0.0052,0.0103,-123,-176,-53
6,amp040_dedupe_same_pitch_25ms,54,10207,8765,7456,1309,2751,0.8507,0.7305,0.7860,0.0114,1.5041,0.0486,-0.0208,0.0102,-795,-583,212
7,amp040_min_duration_30ms,54,10207,8765,7456,1309,2751,0.8507,0.7305,0.7860,0.0114,1.5523,0.0486,-0.0208,0.0102,-795,-583,212
18,combo_amp040_min30_dedupe25,54,10207,8765,7456,1309,2751,0.8507,0.7305,0.7860,0.0114,1.5013,0.0486,-0.0208,0.0102,-795,-583,212
21,combo_ghost_amp040_local_harmonic_dedupe,54,10207,8765,7456,1309,2751,0.8507,0.7305,0.7860,0.0114,1.5997,0.0486,-0.0208,0.0102,-795,-583,212
27,ghost_local_amp040_ratio035,54,10207,8765,7456,1309,2751,0.8507,0.7305,0.7860,0.0114,1.6108,0.0486,-0.0208,0.0102,-795,-583,212
31,prev_best_amp040,54,10207,8765,7456,1309,2751,0.8507,0.7305,0.7860,0.0114,1.5306,0.0486,-0.0208,0.0102,-795,-583,212
14,bp_onset050_frame025_amp035,54,10207,9345,7683,1662,2524,0.8222,0.7527,0.7859,0.0114,1.5763,0.0201,0.0015,0.0101,-215,-230,-15


## 9. Diagnostic plots

These are lightweight plots to quickly see precision/recall tradeoffs. The CSVs above are the source of truth.


In [ ]:
import matplotlib.pyplot as plt

if not summary_df.empty:
    plot_df = summary_df.sort_values('pitch_f1', ascending=True)

    plt.figure(figsize=(10, max(4, 0.4 * len(plot_df))))
    plt.barh(plot_df['variant'], plot_df['pitch_f1'])
    plt.xlabel('Pitch/onset F1')
    plt.title('Basic Pitch note-capture F1 by variant')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 6))
    plt.scatter(summary_df['pitch_recall'], summary_df['pitch_precision'])
    for _, row in summary_df.iterrows():
        plt.annotate(row['variant'], (row['pitch_recall'], row['pitch_precision']), fontsize=8)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision vs recall by Basic Pitch variant')
    plt.tight_layout()
    plt.show()
else:
    print('Run the experiment cell first to create summary_df.')



## 10. Error inspection helpers

Use these after running the experiment to inspect what a variant is missing or hallucinating.


In [ ]:
def show_record_variant_errors(recording, variant, n=30):
    print(f'Recording: {recording} | Variant: {variant}')
    print('\nFalse positives: predicted by Basic Pitch but no GT pitch/onset match')
    if false_pos_df.empty:
        print('No false_pos_df available.')
    else:
        fp = false_pos_df[(false_pos_df['recording'] == recording) & (false_pos_df['variant'] == variant)].copy()
        cols = [c for c in ['start', 'duration', 'midi', 'note_name', 'amplitude'] if c in fp.columns]
        display(fp[cols].sort_values(['start', 'midi']).head(n))

    print('\nMissed GT notes: in GuitarSet but not captured by Basic Pitch')
    if missed_df.empty:
        print('No missed_df available.')
    else:
        miss = missed_df[(missed_df['recording'] == recording) & (missed_df['variant'] == variant)].copy()
        cols = [c for c in ['start', 'duration', 'midi', 'pitch_class', 'true_string', 'true_fret'] if c in miss.columns]
        display(miss[cols].sort_values(['start', 'midi']).head(n))


def show_best_and_worst_records(variant=None, n=10):
    if metrics_df.empty:
        print('Run experiments first.')
        return
    df = metrics_df.copy()
    if variant is not None:
        df = df[df['variant'] == variant]
    print('Best records by pitch_f1:')
    display(df.sort_values('pitch_f1', ascending=False).head(n)[['recording', 'variant', 'n_gt_notes', 'n_pred_notes', 'pitch_precision', 'pitch_recall', 'pitch_f1']])
    print('\nWorst records by pitch_f1:')
    display(df.sort_values('pitch_f1', ascending=True).head(n)[['recording', 'variant', 'n_gt_notes', 'n_pred_notes', 'pitch_precision', 'pitch_recall', 'pitch_f1']])

# Example usage after running experiments:
# show_best_and_worst_records('baseline_amp030')
# show_record_variant_errors(recording='00_Jazz3-150-D_comp', variant='baseline_amp030', n=50)



## 11. Optional: quick threshold sweep

Use this to test amplitude thresholds more granularly without editing `EXPERIMENT_VARIANTS` above. This is useful if the baseline has low recall or too many false positives.


In [ ]:
def make_threshold_sweep_variants(thresholds=None, min_midi=40, max_midi=88):
    thresholds = thresholds or [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
    return [
        {
            'name': f'threshold_{t:.2f}'.replace('.', 'p'),
            'amplitude_threshold': float(t),
            'min_midi': min_midi,
            'max_midi': max_midi,
        }
        for t in thresholds
    ]

# Uncomment to run only threshold sweep variants.
# EXPERIMENT_VARIANTS = make_threshold_sweep_variants()
# Then rerun the experiment suite cell.



## V2 helper summaries

In [ ]:

def show_top_variants(summary_df, min_recall=None, min_precision=None, n=10):
    if summary_df.empty:
        print('Run the experiment suite first.')
        return
    df = summary_df.copy()
    if min_recall is not None:
        df = df[df['pitch_recall'] >= min_recall]
    if min_precision is not None:
        df = df[df['pitch_precision'] >= min_precision]
    display(df.sort_values(['pitch_f1', 'pitch_recall', 'pitch_precision'], ascending=False).head(n).round(4))


def compare_against_previous_best(summary_df, previous_best='prev_best_amp040'):
    if summary_df.empty or previous_best not in set(summary_df['variant']):
        print(f'Could not find {previous_best} in summary_df.')
        return pd.DataFrame()
    base = summary_df[summary_df['variant'] == previous_best].iloc[0]
    out = summary_df.copy()
    for col in ['pitch_precision', 'pitch_recall', 'pitch_f1', 'n_pred_notes', 'n_false_positives', 'n_missed_gt_notes']:
        out[f'delta_{col}_vs_{previous_best}'] = out[col] - base[col]
    out = out.sort_values('delta_pitch_f1_vs_' + previous_best, ascending=False)
    display(out.round(4))
    return out


def summarize_by_record_type(metrics_df):
    if metrics_df.empty:
        print('Run experiments first.')
        return pd.DataFrame()
    tmp = metrics_df.copy()
    tmp['record_type'] = np.where(tmp['is_solo'], 'solo', np.where(tmp['is_comp'], 'comp', 'other'))
    rows = []
    for (variant, record_type), g in tmp.groupby(['variant', 'record_type']):
        n_pred = int(g['n_pred_notes'].sum())
        n_gt = int(g['n_gt_notes'].sum())
        n_match = int(g['n_pitch_onset_matches'].sum())
        precision = n_match / n_pred if n_pred else 0.0
        recall = n_match / n_gt if n_gt else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if precision + recall else 0.0
        rows.append({'variant': variant, 'record_type': record_type, 'recordings': g['recording'].nunique(), 'n_gt_notes': n_gt, 'pitch_precision': precision, 'pitch_recall': recall, 'pitch_f1': f1})
    out = pd.DataFrame(rows).sort_values(['record_type', 'pitch_f1'], ascending=[True, False])
    display(out.round(4))
    out.to_csv(OUTPUT_DIR / 'basic_pitch_variant_summary_by_record_type.csv', index=False)
    return out

print('Top by F1:')
show_top_variants(summary_df, n=10)
print('\nTop while requiring recall >= baseline-ish 0.75:')
show_top_variants(summary_df, min_recall=0.75, n=10)
print('\nComparison vs previous best amp 0.40:')
v2_vs_prev_best_df = compare_against_previous_best(summary_df, previous_best='prev_best_amp040')
print('\nSolo/comp breakdown:')
summary_by_type_df = summarize_by_record_type(metrics_df)


## What to send back for interpretation

After running this V3 notebook, send these CSVs from `OUTPUT_DIR`:

- `basic_pitch_variant_summary_weighted.csv`
- `basic_pitch_variant_comparison_vs_baseline.csv`
- `basic_pitch_variant_metrics_by_record.csv`
- `basic_pitch_onset_tolerance_sweep.csv`
- `basic_pitch_false_positives.csv`
- `basic_pitch_missed_gt_notes.csv`
- `basic_pitch_variant_summary_by_record_type.csv` if the helper cell runs

Best first run:

```python
USE_HELDOUT_SPLIT = True
MAX_AUDIO_RECORDINGS = None
VARIANT_SET = 'core'
```

For final defensible reporting, choose variants on validation, then report the final result on the held-out test split.
